# Controlling MODFLOW 6 with the API — D. Reverse drain

This is part D of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle and the manual solver loop, and [**C. Adjusting recharge with a callback**](modflow-api-C.ipynb) for the callback mechanism this example builds on.

A normal Drain (DRN) package *removes* water from a cell when the head rises above the drain elevation. A **reverse drain** does the opposite: it *injects* water when the head is below the elevation, driving the head up toward it,

$$q = \mathrm{COND}\,(\mathrm{ELEV} - h) \quad\text{when } h < \mathrm{ELEV},\qquad 0 \text{ otherwise.}$$

A standard DRN package cannot be reversed through the API because it recomputes its own matrix contributions each time it assembles the equations. Instead, use the **API package** (`ModflowGwfapi`), whose `HCOF` and `RHS` contributions (the head-coefficient and right-hand-side terms MODFLOW assembles for each cell) are supplied by the user and are *not* overwritten. Each outer iteration a callback writes the head-dependent reverse-drain term as `HCOF = -COND`, `RHS = -COND*ELEV` (or zero), under-relaxed (moved only part of the way toward the new value each iteration) for stability.

**The key idea.** Here the reverse-drain elevations are set to the *analytical water table* of a 1-D unconfined aquifer with uniform recharge and a river (fixed-head) boundary. Holding the water table on that profile, the reverse drain must inject water at exactly the recharge rate that produced it — so the reverse drain *reproduces the recharge* without a Recharge package. The simulated water table and the recovered recharge are both checked against the analytical solution.

Run the imports and library-location cells below first.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and numerical libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from flopy.plot.styles import styles
from modflowapi import Callbacks, run_simulation
from notebook_helpers import find_mf6_libraries

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## The analytical benchmark: a Boussinesq water table

Consider a 1-D unconfined aquifer between a groundwater **divide** (a no-flow boundary, like the crest of a hill) and a **river** (a fixed-head boundary). Uniform recharge $N$ (rainfall) falls on it, so the water table mounds up between the two boundaries. Under the Dupuit-Forchheimer assumptions the steady water table solves the 1-D Boussinesq equation

$$\frac{d}{dx}\!\left(K\,b\,\frac{db}{dx}\right) = -N,$$

where $b = h - z_\text{bot}$ is the saturated thickness. With the divide at $x=0$ ($db/dx = 0$) and the river at $x = L$ (head fixed, saturated thickness $b_L$), the solution is a parabola in $b^2$:

$$b(x)^2 = b_L^2 + \frac{N}{K}\left(L^2 - x^2\right).$$

The model is a 1-D cross section aligned with the model **columns** (1 layer, 1 row, many columns): the divide is the no-flow left edge at $x=0$, and the river is a constant-head cell (head $0$) at the final column. The grid is discretized finely so the numerical water table can match the analytical parabola.

In [ ]:
sim_name = "reverse-drain"
model_name = "rdrn"
workspace = pl.Path(f"models/{sim_name}_api_D")

# 1-D cross section along the columns: 1 layer, 1 row, ncol columns (fine grid)
nlay, nrow, ncol = 1, 1, 200
delr = 5.0  # column width (m) -> 1000 m long cross section
delc = 10.0  # row width (m)
top = 10.0  # land surface (m)
botm = -2.0  # aquifer base (m); below the river head so the river cell stays wet

k = 1.0  # hydraulic conductivity (m/d)
recharge = 1.0e-4  # areal recharge the water table is in equilibrium with (m/d)
river_head = 0.0  # constant head at the river (last column)

cond = 1.0e3  # reverse-drain conductance (large -> heads pin to the elevations)
damping = 0.5  # under-relaxation of the reverse-drain term each outer iteration

# cell-center x, measured from the divide at the left edge (x = 0)
xcenters = (np.arange(ncol) + 0.5) * delr
x_river = xcenters[-1]  # the river (constant head) sits at the last cell

# steady 1-D Boussinesq (Dupuit) water table:
#   b(x)^2 = b_river^2 + (N/K)(x_river^2 - x^2),  b = head - botm
b_river = river_head - botm
analytical_head = botm + np.sqrt(
    b_river**2 + (recharge / k) * (x_river**2 - xcenters**2)
)

# the reverse drain is applied to every cell except the river cell; its elevation
# is the analytical water table there. With all cells active the reduced node
# numbers (1-based) equal the natural node order.
active_cols = np.arange(ncol - 1)
active_nodes = active_cols + 1
elevations = analytical_head[active_cols]

print(f"cross section length : {ncol * delr:.0f} m in {ncol} columns")
print(f"max analytical head  : {analytical_head.max():.3f} m at the divide")
print(f"river head           : {river_head:.3f} m at x = {x_river:.1f} m")

### Build the model

One steady-state GWF model on the fine 1-D grid: no-flow at the divide (the default left boundary), a constant-head river at the last column, and — crucially — **no Recharge package**. The recharge is supplied entirely by the reverse drain, implemented as an API package whose `HCOF`/`RHS` the callback fills in at run time. The initial heads are set to the analytical profile so the solver starts near the answer.

In [ ]:
def build_model():
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=workspace,
        exe_name=str(mf6_exe),
    )
    flopy.mf6.ModflowTdis(
        sim,
        nper=1,
        perioddata=[(1.0, 1, 1.0)],
        time_units="days",
    )
    flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        outer_maximum=2000,
        outer_dvclose=1.0e-8,
        inner_maximum=100,
        inner_dvclose=1.0e-10,
        linear_acceleration="BICGSTAB",
    )
    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=model_name,
        save_flows=True,
    )
    flopy.mf6.ModflowGwfdis(
        gwf,
        length_units="meters",
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=[botm],
    )
    # unconfined (convertible) so transmissivity follows the water table
    flopy.mf6.ModflowGwfnpf(
        gwf,
        icelltype=1,
        k=k,
        save_specific_discharge=True,
    )
    flopy.mf6.ModflowGwfic(
        gwf,
        strt=analytical_head.reshape(nlay, nrow, ncol),
    )
    # river: constant head at the last column
    flopy.mf6.ModflowGwfchd(
        gwf,
        stress_period_data=[[(0, 0, ncol - 1), river_head]],
        pname="river",
    )
    # the reverse drain is supplied through the API package: the callback writes
    # its HCOF and RHS each outer iteration and MF6 does not overwrite them
    flopy.mf6.ModflowGwfapi(
        gwf,
        pname="rdrn",
        maxbound=active_nodes.size,
        print_flows=True,
    )
    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{model_name}.hds",
        budget_filerecord=f"{model_name}.cbc",
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )
    return sim


sim = build_model()
sim.write_simulation(silent=True)
print("wrote simulation to", workspace)

## Drive the reverse drain through the API

The callback wires the reverse drain into the solver. Once, at `initialize`, it assigns the drain to the active cells by writing the API package's `NODELIST` and `NBOUND`. Then, at the start of every outer (Picard) iteration, it reads the current heads, forms the head-dependent reverse-drain term (`HCOF = -COND`, `RHS = -COND*ELEV` where the head is below the elevation, zero elsewhere), and writes it into the package's live `HCOF`/`RHS` arrays — under-relaxed by `damping` so the head-dependent switch does not oscillate. `run_simulation` then drives the whole model, calling back at each iteration.

In [ ]:
MNAME = model_name.upper()


def reverse_drain(sim, step):
    """Supply the reverse-drain HCOF/RHS to the API package each outer iteration."""
    mf6 = sim.mf6
    if step == Callbacks.initialize:
        # assign the reverse drains to the active cells (constant for the run)
        nbound = mf6.get_value_ptr(mf6.get_var_address("NBOUND", MNAME, "RDRN"))
        nodelist = mf6.get_value_ptr(mf6.get_var_address("NODELIST", MNAME, "RDRN"))
        nbound[0] = active_nodes.size
        nodelist[:] = active_nodes

    if step == Callbacks.iteration_start:
        head = mf6.get_value_ptr(mf6.get_var_address("X", MNAME))
        hcof = mf6.get_value_ptr(mf6.get_var_address("HCOF", MNAME, "RDRN"))
        rhs = mf6.get_value_ptr(mf6.get_var_address("RHS", MNAME, "RDRN"))
        h = head[active_nodes - 1]
        injecting = h < elevations
        new_hcof = np.where(injecting, -cond, 0.0)
        new_rhs = np.where(injecting, -cond * elevations, 0.0)
        # under-relax (damp) the term before each solve for stability
        hcof[:] = (1.0 - damping) * hcof + damping * new_hcof
        rhs[:] = (1.0 - damping) * rhs + damping * new_rhs

In [ ]:
run_simulation(lib_name, workspace, reverse_drain, verbose=False)

## Compare to the analytical solution

Read the simulated heads back from the head file, then recover the reverse-drain inflow at each cell from the converged heads, $q = \mathrm{COND}\,(\mathrm{ELEV}-h)$. Two things should hold if the discretization is fine enough:

1. the simulated **water table** lies on the analytical Boussinesq parabola, and
2. the reverse-drain **inflow per unit area** equals the specified recharge $N$ — the drain has *reproduced the recharge* that the analytical water table was built from.

In [ ]:
gwf = sim.get_model(model_name)
h_sim = gwf.output.head().get_data().flatten()  # one head per column

# reverse-drain inflow recovered from the converged heads
q_drn = cond * np.maximum(elevations - h_sim[active_cols], 0.0)  # m^3/d per cell
cell_area = delr * delc
inferred_recharge = q_drn / cell_area  # m/d per cell (should ~ recharge)

total_inflow = q_drn.sum()
total_recharge = recharge * cell_area * active_nodes.size

max_head_err = np.max(np.abs(h_sim[active_cols] - elevations))
print(f"max |simulated - analytical| head : {max_head_err:.2e} m")
print(f"total reverse-drain inflow        : {total_inflow:.4f} m^3/d")
print(f"total specified recharge (N*area) : {total_recharge:.4f} m^3/d")
print(f"inferred recharge (mean)          : {inferred_recharge.mean() * 1e3:.4f} mm/d")
print(f"specified recharge                : {recharge * 1e3:.4f} mm/d")

In [ ]:
xa = xcenters[active_cols]

with styles.USGSPlot():
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(8, 6.5), sharex=True, constrained_layout=True
    )

    # --- A. water table ---
    ax1.fill_between(xcenters, botm, top, color="0.93", zorder=0)
    ax1.plot(
        xcenters, analytical_head, color="0.2", lw=2.0, label="Analytical (Boussinesq)"
    )
    ax1.plot(
        xcenters[::8],
        h_sim[::8],
        ls="none",
        marker="o",
        ms=5,
        mfc="none",
        mec="tab:red",
        mew=1.2,
        label="MODFLOW (reverse drain)",
    )
    ax1.axhline(botm, color="saddlebrown", lw=1.2, ls="--", label="Aquifer base")
    ax1.plot(
        [x_river],
        [river_head],
        marker="v",
        ms=11,
        mfc="tab:blue",
        mec="k",
        mew=0.6,
        clip_on=False,
        zorder=6,
        label="River (constant head)",
    )
    ax1.annotate(
        "divide\n(no flow)",
        xy=(0, analytical_head[0]),
        xytext=(60, analytical_head[0] - 2.0),
        fontsize=9,
        arrowprops=dict(arrowstyle="->", lw=0.8),
    )
    ax1.set_ylim(botm - 0.5, top)
    styles.heading(ax=ax1, letter="A", heading="Water table")
    styles.ylabel(ax=ax1, label="Head (m)")
    ax1.legend(loc="upper right", frameon=False, fontsize=9)

    # --- B. recharge recovered by the reverse drain ---
    ax2.plot(
        xa,
        inferred_recharge * 1e3,
        color="tab:red",
        lw=1.5,
        label="Reverse-drain inflow / area",
    )
    ax2.axhline(
        recharge * 1e3, color="0.2", lw=2.0, ls="--", label="Specified recharge $N$"
    )
    ax2.set_ylim(0, recharge * 1e3 * 1.6)
    styles.heading(
        ax=ax2, letter="B", heading="Recharge reproduced by the reverse drain"
    )
    styles.xlabel(ax=ax2, label="Distance from divide, x (m)")
    styles.ylabel(ax=ax2, label="Recharge (mm/d)")
    ax2.legend(loc="lower center", frameon=False, fontsize=9)

    fig.align_ylabels((ax1, ax2))
    fig.savefig(fig_path / "reverse-drain-water-table.png", dpi=200)

**What to look for.** In panel A the red MODFLOW markers sit on the black analytical parabola — the reverse drain holds the water table on the Boussinesq profile, mounding up from the river toward the divide. In panel B the inflow the drain had to supply, divided by cell area, is essentially the flat specified recharge $N$: the reverse drain *discovered* the recharge that produced the water table, cell by cell, and its total matches $N$ times the recharge area. The small departures near the boundaries shrink as the grid is refined.

## Recap

This example used the API package to add a boundary MODFLOW 6 will not apply on its own — a head-dependent *inflow* — by writing its `HCOF` and `RHS` directly and under-relaxing the head-dependent switch each outer iteration. The same lifecycle and the same `get_value_ptr` memory access from the earlier notebooks let you supply a custom conductance boundary and verify it against a classical analytical solution.